# Assignment No. - 2
**Name:** Kadam Abhijeet Jalindar &nbsp;&nbsp;&nbsp;&nbsp; **Roll No.:** COA426 &nbsp;&nbsp;&nbsp;&nbsp; **Div:** A

**Title of the Assignment:** Binary classification using Deep Neural Networks Example: Classify movie reviews into "positive" reviews and "negative" reviews, just based on the text content of the reviews. Use IMDB dataset.

## Domain & Context

- **DOMAIN:** Digital content and entertainment industry
- **CONTEXT:** The objective of this project is to build a text classification model that analyses the customer's sentiments based on their reviews in the IMDB database. The model uses a complex deep learning model to build an embedding layer followed by a classification algorithm to analyse the sentiment of the customers.
- **DATA DESCRIPTION:** The dataset contains 50,000 movie reviews from IMDB, labelled by sentiment (positive/negative). Reviews have been preprocessed, and each review is encoded as a sequence of word indexes (integers). The words are indexed by their frequency in the dataset, meaning the word that has index 1 is the most frequent word.
- **PROJECT OBJECTIVE:** Build a sequential NLP classifier which can use input text parameters to determine the customer sentiments.

## Dataset Description

- The IMDB sentiment classification dataset consists of 50,000 movie reviews from IMDB users that are labeled as either positive (1) or negative (0).
- The reviews are preprocessed and each one is encoded as a sequence of word indexes in the form of integers.
- The words within the reviews are indexed by their overall frequency within the dataset.
- For example, the integer "2" encodes the second most frequent word in the data.
- The 50,000 reviews are split into 25,000 for training and 25,000 for testing.
- Use the first 20 words from each review to speed up training, using a max vocabulary size of 10,000.
- As a convention, "0" does not stand for a specific word, but instead is used to encode any unknown word.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [ ]:
# Loading IMDB data with most frequent 10,000 words
# You may take top 10,000 frequently used words in movie reviews; others are discarded.
from keras.datasets import imdb

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=10000)

In [ ]:
# Consolidating data for EDA (Exploratory Data Analysis)
# EDA is used by data scientists to analyze and investigate datasets
# and summarize their main characteristics.
# axis=0 means stacking vertically (row-wise)
data = np.concatenate((X_train, X_test), axis=0)
label = np.concatenate((y_train, y_test), axis=0)

In [ ]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
# series of numbers converted word to vocabulary associated with index
print("Review is ", X_train[0])
print("Label is ", y_train[0])

In [ ]:
# Retrieve the word index file mapping words to indices
vocab = imdb.get_word_index()
print(vocab)

In [ ]:
print("y_train:", y_train)
print("y_test:", y_test)

In [ ]:
print("Categories:", np.unique(label))
print("Number of unique words:", len(np.unique(np.hstack(data))))
# The hstack() function is used to stack arrays in sequence horizontally (column wise).

In [ ]:
length = [len(i) for i in data]
print("Average Review length:", np.mean(length))
print("Standard Deviation:", round(np.std(length)))
# The whole dataset contains 9998 unique words and the average review length is 234 words,
# with a standard deviation of 173 words.

In [ ]:
# If you look at the data you will realize it has been already pre-processed.
# All words have been mapped to integers and the integers represent the words sorted by their frequency.
# This is very common in text analysis to represent a dataset like this.
# So 4 represents the 4th most used word, 5 the 5th most used word and so on...
# The integer 1 is reserved for the start marker,
# the integer 2 for an unknown word and 0 for padding.

# Let's look at a single training example:
print("Label:", label[0])
print("Label:", label[1])

In [ ]:
print(data[0])

# Retrieves a dict mapping words to their index in the IMDB dataset.
index = imdb.get_word_index()  # word to index

# Create inverted index from a dictionary with document ids as keys
# and a list of terms as values for each document
reverse_index = dict([(value, key) for (key, value) in index.items()])  # id to word

decoded = " ".join([reverse_index.get(i - 3, "#") for i in data[0]])
# The indices are offset by 3 because 0, 1 and 2 are reserved indices
# for "padding", "start of sequence" and "unknown".
print(decoded)

In [ ]:
# Vectorization is the process of converting textual data into numerical vectors
# and is a process that is usually applied once the text is cleaned.

# Now it is time to prepare our data. We will vectorize every review and fill it with zeros
# so that it contains exactly 10,000 numbers.
# That means we fill every review that is shorter than 10,000 with zeros.
# We do this because the biggest review is nearly that long and every input for our neural
# network needs to have the same size.
# We also transform the targets into floats.

# VECTORIZE as one cannot feed integers into a NN
# Encoding the integer sequences into a binary matrix - one hot encoder basically
# From integers representing words at various lengths - to a normalized one hot encoded tensor
# (matrix) of 10k columns

def vectorize(sequences, dimension=10000):
    # We will vectorize every review and fill it with zeros so that it contains exactly 10,000 numbers.
    # Create an all-zero matrix of shape (len(sequences), dimension)
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1
    return results

In [ ]:
# Now we split our data into a training and a testing set.
# Set a VALIDATION set
test_x = data[:10000]
test_y = label[:10000]
train_x = data[10000:]
train_y = label[10000:]

print("test_x shape:", test_x.shape)
print("test_y shape:", test_y.shape)
print("train_x shape:", train_x.shape)
print("train_y shape:", train_y.shape)

In [ ]:
# Adding sequence to data
data = vectorize(data)
label = np.array(label).astype("float32")

In [ ]:
labelDF = pd.DataFrame({'label': label})
sns.countplot(x='label', data=labelDF)

In [ ]:
# Creating train and test data set
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    data, label, test_size=0.20, random_state=1
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

In [ ]:
# Let's create sequential model
from keras.utils import to_categorical
from keras import models
from keras import layers

model = models.Sequential()

# Input - Layer
# Note that we set the input-shape to 10,000 at the input-layer
# because our reviews are 10,000 integers long.
# The input-layer takes 10,000 as input and outputs it with a shape of 50.
model.add(layers.Dense(50, activation="relu", input_shape=(10000,)))

# Hidden - Layers
# Please note you should always use a dropout rate between 20% and 50%.
# Here in our case 0.3 means 30% dropout.
# We are using dropout to prevent overfitting.
model.add(layers.Dropout(0.3, noise_shape=None, seed=None))
model.add(layers.Dense(50, activation="relu"))
model.add(layers.Dropout(0.2, noise_shape=None, seed=None))
model.add(layers.Dense(50, activation="relu"))

# Output - Layer
model.add(layers.Dense(1, activation="sigmoid"))

model.summary()

In [ ]:
# For early stopping
# Stop training when a monitored metric has stopped improving.
# monitor: Quantity to be monitored.
# patience: Number of epochs with no improvement after which training will be stopped.
import tensorflow as tf

callback = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=3)

In [ ]:
# We use the "adam" optimizer, an algorithm that changes the weights and biases during training.
# We also choose binary-crossentropy as loss (because we deal with binary classification)
# and accuracy as our evaluation metric.
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
results = model.fit(
    X_train, y_train,
    epochs=2,
    batch_size=500,
    validation_data=(X_test, y_test),
    callbacks=[callback]
)

In [ ]:
# Let's check mean accuracy of our model
print(np.mean(results.history["val_accuracy"]))

In [ ]:
# Evaluate the model
score = model.evaluate(X_test, y_test, batch_size=500)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

In [ ]:
# Let's plot training history of our model

# list all data in history
print(results.history.keys())

# summarize history for accuracy
plt.plot(results.history['accuracy'])
plt.plot(results.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

In [ ]:
# summarize history for loss
plt.plot(results.history['loss'])
plt.plot(results.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()